# Fig 2G — Regulatory Map (Sankey)

Three-column flow: perturbation targets (left) → gene programs (middle) → phenotype (right).  
Blue = activation axis (KO → HLA-low);  Red = repression axis (KO → HLA-high).

**Illustrator-friendly export via kaleido**: PDF and SVG are saved with embedded fonts / vector paths so the figure can be directly opened and edited in Adobe Illustrator.

In [ ]:
import os
import plotly.graph_objects as go
import plotly.io as pio

OUT_DIR = '../figures/nature_figures/Fig2'
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Node & link definitions

In [ ]:
def columnize(gene_list, per_row=2, sep='   '):
    lines = []
    for i in range(0, len(gene_list), per_row):
        lines.append(sep.join(gene_list[i:i+per_row]))
    return '<br>'.join(lines)

t_labels = {
    'TC0': '<b>TC0: Retromer/Recycling</b><br>'  + columnize(['VPS35', 'VPS29', 'B2M', 'FOSL1', 'RBM10', 'WWP2', 'SETD2', 'C1GALT1C1']),
    'TC1': '<b>TC1: Proteostasis/UPS</b><br>'    + columnize(['COPS3', 'COPS5', 'COPS8', 'AATF', 'DUSP4', 'INTS1', 'PSMG4', 'SNRNP40']),
    'TC2': '<b>TC2: Epigenetic Hub</b><br>'      + columnize(['CIITA', 'IRF1', 'RFX5', 'RFXANK', 'RFXAP', 'NLRC5', 'GATA2', 'EZH2']),
    'TC3': '<b>TC3: Biosynthesis</b><br>'        + columnize(['ALG2', 'TBP', 'CDC123', 'RABGGTA', 'HARS1', 'IARS1', 'RPP21', 'RTCB']),
    'TC4': '<b>TC4: QC/Folding</b><br>'          + columnize(['CALR', 'TAP2', 'SLC35A1', 'SLC35A2', 'GALNT1', 'NCSTN', 'TSPAN4', 'USP41']),
    'TC5': '<b>TC5: Signaling Core</b><br>'      + columnize(['STAT1', 'JAK1', 'JAK2', 'IFNGR1', 'IFNGR2', 'SRPRA', 'SRP14', 'SRP19']),
    'TC6': '<b>TC6: Golgi Machinery</b><br>'     + columnize(['COG1', 'COG3', 'COG4', 'COG8', 'SYS1', 'ARF4', 'GET1', 'USO1']),
}

f_labels = {
    'FC1': '<b>FC1: ER Stress</b><br>'      + columnize(['MANF', 'HSPA5', 'CALU', 'CANX']),
    'FC2': '<b>FC2: ISR Stress</b><br>'     + columnize(['ATF4', 'ASNS', 'DDIT3', 'DDIT4']),
    'FC4': '<b>FC4: Trafficking</b><br>'    + columnize(['GOLGA2', 'RAB1A', 'KDELR2', 'SEC61A1']),
    'FC5': '<b>FC5: Proliferation</b><br>'  + columnize(['MKI67', 'CDC5L', 'CCNB1', 'TOP2A']),
    'FC6': '<b>FC6: Cytoskeletal</b><br>'   + columnize(['ACTB', 'TUBA1B', 'TMSB4X', 'CFL1']),
    'FC7': '<b>FC7: HLA Program</b><br>'    + columnize(['BST2', 'GBP1', 'HLA-A', 'HLA-B', 'HLA-C']),
    'FC8': '<b>FC8: Chromatin</b><br>'      + columnize(['HMGB2', 'BRD2', 'BRD4', 'DAXX']),
}

p_label = '<b>PHENOTYPE:<br>SURFACE HLA</b>'

all_labels = list(t_labels.values()) + list(f_labels.values()) + [p_label]
idx = {label: i for i, label in enumerate(all_labels)}

# (source, target, value, color)
C_BLUE_STRONG = 'rgba(0,0,255,0.4)'
C_BLUE_LIGHT  = 'rgba(0,0,255,0.25)'
C_RED_STRONG  = 'rgba(255,0,0,0.4)'
C_RED_LIGHT   = 'rgba(255,0,0,0.25)'

links = [
    # Stage 1 — Target cluster → Feature cluster
    (t_labels['TC5'], f_labels['FC7'], 0.45, C_BLUE_STRONG),
    (t_labels['TC5'], f_labels['FC5'], 0.12, C_BLUE_LIGHT ),
    (t_labels['TC3'], f_labels['FC2'], 0.19, C_BLUE_STRONG),
    (t_labels['TC3'], f_labels['FC7'], 0.29, C_BLUE_LIGHT ),
    (t_labels['TC1'], f_labels['FC5'], 0.21, C_BLUE_STRONG),
    (t_labels['TC2'], f_labels['FC8'], 0.28, C_RED_STRONG ),
    (t_labels['TC2'], f_labels['FC7'], 0.31, C_RED_STRONG ),
    (t_labels['TC4'], f_labels['FC1'], 0.24, C_RED_STRONG ),
    (t_labels['TC4'], f_labels['FC7'], 0.19, C_RED_LIGHT  ),
    (t_labels['TC6'], f_labels['FC4'], 0.26, C_RED_STRONG ),
    (t_labels['TC6'], f_labels['FC7'], 0.29, C_RED_STRONG ),
    (t_labels['TC6'], f_labels['FC1'], 0.11, C_RED_LIGHT  ),
    (t_labels['TC0'], f_labels['FC6'], 0.22, C_RED_STRONG ),
    (t_labels['TC0'], f_labels['FC7'], 0.18, C_RED_LIGHT  ),
    # Stage 2 — Feature cluster → Phenotype
    (f_labels['FC7'], p_label, 0.50, C_BLUE_LIGHT ),
    (f_labels['FC2'], p_label, 0.22, C_BLUE_LIGHT ),
    (f_labels['FC5'], p_label, 0.15, C_BLUE_LIGHT ),
    (f_labels['FC8'], p_label, 0.30, C_RED_LIGHT  ),
    (f_labels['FC1'], p_label, 0.25, C_RED_LIGHT  ),
    (f_labels['FC4'], p_label, 0.35, C_RED_LIGHT  ),
    (f_labels['FC6'], p_label, 0.20, C_RED_LIGHT  ),
]

## 2. Build Sankey

In [ ]:
fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(pad=30, thickness=22, label=all_labels, color='whitesmoke',
              line=dict(color='black', width=0.5)),
    link=dict(
        source=[idx[l[0]] for l in links],
        target=[idx[l[1]] for l in links],
        value =[l[2]      for l in links],
        color =[l[3]      for l in links],
    )
)])

fig.update_layout(
    title_text=' ',
    font=dict(family='Arial, Helvetica, sans-serif', size=11),
    width=1100, height=1100,
    margin=dict(t=120, l=60, r=60, b=80),
    paper_bgcolor='white', plot_bgcolor='white',
)

# Column headers
fig.add_annotation(x=-0.03, y=1.03,  xref='paper', yref='paper', showarrow=False,
                   text='<b>Target Clusters (TC)</b>',  font=dict(size=14))
fig.add_annotation(x=0.51,  y=1.03,  xref='paper', yref='paper', showarrow=False,
                   text='<b>Feature Clusters (FC)</b>', font=dict(size=14))
fig.add_annotation(x=1.01,  y=1.03,  xref='paper', yref='paper', showarrow=False,
                   text='<b>Phenotype</b>',             font=dict(size=14))

# Title + legend
fig.add_annotation(x=-0.03, y=1.10,  xref='paper', yref='paper', showarrow=False,
                   text='<b>Complete HLA Regulatory Network</b>',
                   font=dict(size=18), xanchor='left')
fig.add_annotation(x=1.01, y=-0.03, xref='paper', yref='paper', showarrow=False,
                   text='<b>BLUE:</b> Activation axis (KO → HLA Low)',
                   font=dict(color='#2E3192', size=11), xanchor='right')
fig.add_annotation(x=1.01, y=-0.06, xref='paper', yref='paper', showarrow=False,
                   text='<b>RED:</b> Repression axis (KO → HLA High)',
                   font=dict(color='#ED1C24',  size=11), xanchor='right')


# ── Print-friendly text listing for copy/paste ────────────────────
print('\n' + '='*70)
print('FIG 2G  —  Regulatory map text content')
print('='*70)

print('\nTitle: Complete HLA Regulatory Network')
print('\nColumn headers:')
print('  Left:   Target Clusters (TC)')
print('  Middle: Feature Clusters (FC)')
print('  Right:  Phenotype')

print('\nLegend:')
print('  BLUE: Activation axis (KO → HLA Low)')
print('  RED:  Repression axis (KO → HLA High)')

print('\n' + '-'*70)
print('TARGET CLUSTERS (left column)')
print('-'*70)
tc_full = {
    'TC0': ('Retromer/Recycling', ['VPS35', 'VPS29', 'B2M', 'FOSL1', 'RBM10', 'WWP2', 'SETD2', 'C1GALT1C1']),
    'TC1': ('Proteostasis/UPS',   ['COPS3', 'COPS5', 'COPS8', 'AATF', 'DUSP4', 'INTS1', 'PSMG4', 'SNRNP40']),
    'TC2': ('Epigenetic Hub',     ['CIITA', 'IRF1', 'RFX5', 'RFXANK', 'RFXAP', 'NLRC5', 'GATA2', 'EZH2']),
    'TC3': ('Biosynthesis',       ['ALG2', 'TBP', 'CDC123', 'RABGGTA', 'HARS1', 'IARS1', 'RPP21', 'RTCB']),
    'TC4': ('QC/Folding',         ['CALR', 'TAP2', 'SLC35A1', 'SLC35A2', 'GALNT1', 'NCSTN', 'TSPAN4', 'USP41']),
    'TC5': ('Signaling Core',     ['STAT1', 'JAK1', 'JAK2', 'IFNGR1', 'IFNGR2', 'SRPRA', 'SRP14', 'SRP19']),
    'TC6': ('Golgi Machinery',    ['COG1', 'COG3', 'COG4', 'COG8', 'SYS1', 'ARF4', 'GET1', 'USO1']),
}
for k, (name, genes) in tc_full.items():
    print(f'  {k}: {name}')
    print(f'    {", ".join(genes)}')

print('\n' + '-'*70)
print('FEATURE CLUSTERS (middle column)')
print('-'*70)
fc_full = {
    'FC1': ('ER Stress',     ['MANF', 'HSPA5', 'CALU', 'CANX']),
    'FC2': ('ISR Stress',    ['ATF4', 'ASNS', 'DDIT3', 'DDIT4']),
    'FC4': ('Trafficking',   ['GOLGA2', 'RAB1A', 'KDELR2', 'SEC61A1']),
    'FC5': ('Proliferation', ['MKI67', 'CDC5L', 'CCNB1', 'TOP2A']),
    'FC6': ('Cytoskeletal',  ['ACTB', 'TUBA1B', 'TMSB4X', 'CFL1']),
    'FC7': ('HLA Program',   ['BST2', 'GBP1', 'HLA-A', 'HLA-B', 'HLA-C']),
    'FC8': ('Chromatin',     ['HMGB2', 'BRD2', 'BRD4', 'DAXX']),
}
for k, (name, genes) in fc_full.items():
    print(f'  {k}: {name}')
    print(f'    {", ".join(genes)}')

print('\n' + '-'*70)
print('PHENOTYPE (right column)')
print('-'*70)
print('  PHENOTYPE: SURFACE HLA')

print('\n' + '-'*70)
print('LINKS (source → target, value, axis)')
print('-'*70)
edges_readable = [
    ('TC5 Signaling Core',       'FC7 HLA Program',    0.45, 'blue'),
    ('TC5 Signaling Core',       'FC5 Proliferation',  0.12, 'blue'),
    ('TC3 Biosynthesis',         'FC2 ISR Stress',     0.19, 'blue'),
    ('TC3 Biosynthesis',         'FC7 HLA Program',    0.29, 'blue'),
    ('TC1 Proteostasis/UPS',     'FC5 Proliferation',  0.21, 'blue'),
    ('TC2 Epigenetic Hub',       'FC8 Chromatin',      0.28, 'red'),
    ('TC2 Epigenetic Hub',       'FC7 HLA Program',    0.31, 'red'),
    ('TC4 QC/Folding',           'FC1 ER Stress',      0.24, 'red'),
    ('TC4 QC/Folding',           'FC7 HLA Program',    0.19, 'red'),
    ('TC6 Golgi Machinery',      'FC4 Trafficking',    0.26, 'red'),
    ('TC6 Golgi Machinery',      'FC7 HLA Program',    0.29, 'red'),
    ('TC6 Golgi Machinery',      'FC1 ER Stress',      0.11, 'red'),
    ('TC0 Retromer/Recycling',   'FC6 Cytoskeletal',   0.22, 'red'),
    ('TC0 Retromer/Recycling',   'FC7 HLA Program',    0.18, 'red'),
    ('FC7 HLA Program',          'PHENOTYPE',          0.50, 'blue'),
    ('FC2 ISR Stress',           'PHENOTYPE',          0.22, 'blue'),
    ('FC5 Proliferation',        'PHENOTYPE',          0.15, 'blue'),
    ('FC8 Chromatin',            'PHENOTYPE',          0.30, 'red'),
    ('FC1 ER Stress',            'PHENOTYPE',          0.25, 'red'),
    ('FC4 Trafficking',          'PHENOTYPE',          0.35, 'red'),
    ('FC6 Cytoskeletal',         'PHENOTYPE',          0.20, 'red'),
]
for s, t, v, c in edges_readable:
    arrow = '━━▶' if v >= 0.3 else ('─▶' if v >= 0.2 else '──▶')
    print(f'  [{c.upper():4s}] {s:30s} {arrow} {t:22s}  (v={v:.2f})')

fig.show()

## 3. Export Illustrator-friendly files

`pio.write_image` uses **kaleido** (Chromium-headless) to rasterise / vectorise the plotly figure. Both PDF and SVG preserve text as selectable/editable objects in Illustrator.

In [ ]:
for ext, fmt in [('pdf', 'pdf'), ('svg', 'svg'), ('png', 'png')]:
    path = os.path.join(OUT_DIR, f'Fig2G_regulatory_map.{ext}')
    pio.write_image(fig, path, format=fmt, width=1100, height=1100, scale=2)
    print(f'Saved {path}')

## 4. Illustrator editing

Open `Fig2G_regulatory_map.pdf` (or `.svg`) in Illustrator:

1. **Text is selectable / editable** — gene names, cluster headers, title, legend entries are all real text objects.
2. **Ribbons are vector paths** — click any link, open Appearance panel to change colour/opacity.
3. **Nodes are rectangles** — click-drag to resize or restack.
4. To unlock individual elements: <kbd>Object → Ungroup</kbd> (⌘/Ctrl+Shift+G) once or twice.
5. The SVG version preserves named groups (per Sankey layer) which is convenient for batch-restyling.

**Files written:**
- `../figures/nature_figures/Fig2/Fig2G_regulatory_map.pdf`
- `../figures/nature_figures/Fig2/Fig2G_regulatory_map.svg`
- `../figures/nature_figures/Fig2/Fig2G_regulatory_map.png` (preview)